In [ ]:
%pip install -q -e .. plotly

# 03 — FWI animation

Same discover-and-fetch workflow as `02_fwi_map.ipynb`, but for a window
of days.  Plotly's built-in `animation_frame=` provides the date slider —
no extra dependencies, no callback wiring.

In [ ]:
import httpx

server = "https://edr.example.com"  # replace with your EDR server root

collections = httpx.get(f"{server}/collections").raise_for_status().json()["collections"]
collection_id = collections[0]["id"]
collection_url = f"{server}/collections/{collection_id}"

In [ ]:
import xarray as xr

import edr_xarray  # registers engine="edr"

spain = (-9.5, 36.0, 3.3, 43.8)
ds = xr.open_dataset(collection_url, engine="edr", bbox=spain)

## 1. Pick a window

In [ ]:
print("from:", ds.t.values[0])
print("to:  ", ds.t.values[-1])

In [ ]:
date_from = "2024-08-01"  # CHANGE ME — start of window
date_to   = "2024-08-15"  # CHANGE ME — end of window (inclusive)

## 2. Fetch the window

One HTTP request fetches every frame in the window.  Subsequent slider
scrubs are local — no extra network traffic.

In [ ]:
var = next(iter(ds.data_vars))
frames = ds[var].sel(t=slice(date_from, date_to)).load()

## 3. Animate

In [ ]:
import plotly.express as px

df = frames.to_dataframe(name=var).reset_index().dropna(subset=[var])
df["t"] = df["t"].dt.strftime("%Y-%m-%d")  # plotly slider needs string frames

# EFFIS 6-class FWI danger thresholds, normalised to range_color=(0, 50)
fig = px.density_mapbox(
    df, lat="y", lon="x", z=var,
    animation_frame="t",
    radius=8,
    center={"lat": 40, "lon": -3},  # Spain
    zoom=4,
    mapbox_style="carto-positron",
    color_continuous_scale=[
        (0.000, "#008000"),  # Very low
        (0.104, "#FFFF00"),  # Low      (5.2/50)
        (0.224, "#FFA500"),  # Moderate (11.2/50)
        (0.426, "#FF0000"),  # High     (21.3/50)
        (0.760, "#654321"),  # Very high(38.0/50)
        (1.000, "#000000"),  # Extreme
    ],
    range_color=(0, 50),
    title=f"{var} — drag the slider",
)
fig.update_layout(height=600)
fig